In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [11]:
df = pd.read_csv("../Dataset/StudentPerformanceFactors_Cleaned.csv")
df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


ENCODE CATEGORICAL FEATURES

In [12]:
label_encoder = LabelEncoder()
categorical_columns = df.select_dtypes(include="object").columns

for column in categorical_columns:
    df[column] = label_encoder.fit_transform(df[column])

df.head()

C:\Users\sumit\AppData\Local\Temp\ipykernel_12384\3663785741.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,1,0,0,7,73,1,1,0,1,2,1,2,3,0,1,2,1,67
1,19,64,1,2,0,8,59,1,1,2,2,2,1,0,4,0,0,1,0,61
2,24,98,2,2,1,7,91,2,1,2,2,2,1,1,4,0,2,2,1,74
3,29,89,1,2,1,8,98,2,1,1,2,2,1,0,4,0,1,1,1,71
4,19,92,2,2,1,6,65,2,1,3,2,0,1,1,4,0,0,2,0,70


SELECT FEATURES AND TARGET VARIABLE

In [13]:
X = df.drop("Exam_Score", axis=1)
y = df["Exam_Score"]

SPLIT THE DATASET

In [14]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

FEATURE SCALING

In [15]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("Feature Scaling Completed Successfully.")

Feature Scaling Completed Successfully.


DEFINE THE HYPERPARAMETER GRID

In [16]:
param_grid = {"n_estimators": [100, 200],
              "max_depth": [10, 20, None],
              "min_samples_split": [2, 5],
              "min_samples_leaf": [1, 2]}

PERFORM HYPERPARAMETER TUNING USING GRIDSEARCHCV

In [17]:
grid_search = GridSearchCV(estimator=RandomForestRegressor(random_state=42),
                           param_grid=param_grid,cv=5,scoring="r2",n_jobs=-1)
grid_search.fit(X_train, y_train)
print("Hyperparameter Tuning Completed Successfully.")

Hyperparameter Tuning Completed Successfully.


BEST HYPERPARAMETERS

In [18]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation R² Score:")
print(grid_search.best_score_)

Best Parameters:
{'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}

Best Cross-Validation R² Score:
0.6310557465653114


TRAIN THE OPTIMIZED RANDOM FOREST MODEL

In [19]:
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

EVALUATE THE OPTIMIZED MODEL

In [21]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² Score: {r2:.4f} ({r2*100:.2f}% variance.)")

Mean Absolute Error (MAE): 1.09
Mean Squared Error (MSE): 4.65
Root Mean Squared Error (RMSE): 2.16
R² Score: 0.6713 (67.13% variance.)
